# 06 - ANN & LSTM Benchmark

I used deep learning only as a benchmark after the basic ML model was ready.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from torch import nn

In [ ]:
hourly = pd.read_csv(
    "../01_Data/processed/hourly_energy.csv",
    parse_dates=["date_time"]
)

In [ ]:
y = hourly["energy_kwh"].values.astype("float32")
train_end = int(len(y) * 0.8)

## Scale using the training part

In [ ]:
mu = y[:train_end].mean()
sd = y[:train_end].std()
y_scaled = (y - mu) / sd

## Make 24-hour sequences

In [ ]:
seq_len = 24
X, target, pos = [], [], []

In [ ]:
for i in range(seq_len, len(y_scaled)):
    X.append(y_scaled[i-seq_len:i])
    target.append(y_scaled[i])
    pos.append(i)

In [ ]:
X = np.array(X, dtype="float32")
target = np.array(target, dtype="float32")
pos = np.array(pos)

## LSTM model

In [ ]:
class SmallLSTM(nn.Module):
    def __init__(self):
        super().__init__()
        self.lstm = nn.LSTM(1, 16, batch_first=True)
        self.fc = nn.Linear(16, 1)

    def forward(self, x):
        x, _ = self.lstm(x)
        return self.fc(x[:, -1, :])


In [ ]:
model = SmallLSTM()
loss_fn = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.005)

## Prepare train and test sequences

In [ ]:
mask = pos < train_end
X_train = torch.tensor(X[mask]).unsqueeze(-1)
y_train = torch.tensor(target[mask]).unsqueeze(-1)

In [ ]:
X_test = torch.tensor(X[~mask]).unsqueeze(-1)
y_test = target[~mask]

## Train the LSTM

In [ ]:
for epoch in range(30):
    optimizer.zero_grad()
    pred = model(X_train)
    loss = loss_fn(pred, y_train)
    loss.backward()
    optimizer.step()

In [ ]:
print("Final train loss:", round(float(loss.item()), 4))

## Check LSTM result

In [ ]:
with torch.no_grad():
    pred_lstm = model(X_test).squeeze().numpy()

In [ ]:
pred_lstm = pred_lstm * sd + mu
y_true = y_test * sd + mu

In [ ]:
from sklearn.metrics import mean_absolute_error, r2_score
print("LSTM MAE:", round(mean_absolute_error(y_true, pred_lstm), 3))
print("LSTM R2:", round(r2_score(y_true, pred_lstm), 3))

## LSTM visual

In [ ]:
plt.figure(figsize=(10,4))
plt.plot(y_true[-168:], label="Actual")
plt.plot(pred_lstm[-168:], label="LSTM")
plt.legend()
plt.title("LSTM Benchmark")
plt.ylabel("kWh")
plt.tight_layout()
plt.show()

In [ ]:
plt.savefig("../04_Visuals/forecasting/10_lstm_benchmark.png", dpi=150)
plt.show()


### Note

A small ANN was already tested in the previous notebook. LSTM is useful here because the input is a sequence of previous hourly values. CNN was not used because there is no image input in this project.